# CorrDiff - Fase 9 - Estrutura Espacial do Radar

Correlação espacial, semivariograma, anisotropia e morfologia dos eventos nos patches 32×32.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

OUT = Path('../analysis_outputs/09_spatial_structure')
summary = json.loads((OUT/'analysis_summary.json').read_text())
summary


## 1. Estratos exatos de intensidade

In [ ]:
strata = pd.read_parquet(OUT/'patch_strata_counts.parquet')
display(strata)


## 2. Correlação espacial do target log1p

In [ ]:
spatial = pd.read_parquet(OUT/'spatial_lag_statistics.parquet')
for direction, g in spatial[spatial.field.eq('target_log1p')].groupby('direction'):
    g = g.sort_values('distance_km')
    plt.figure(figsize=(8,4))
    plt.plot(g.distance_km, g.pearson_r, marker='o')
    plt.xlabel('Distância (km)')
    plt.ylabel('Correlação de Pearson')
    plt.title(f'target_log1p - {direction}')
    plt.ylim(-0.05,1.05)
    plt.tight_layout()
    plt.show()


## 3. Semivariograma do target

In [ ]:
for direction, g in spatial[spatial.field.eq('target_log1p')].groupby('direction'):
    g = g.sort_values('distance_km')
    plt.figure(figsize=(8,4))
    plt.plot(g.distance_km, g.semivariance, marker='o')
    plt.xlabel('Distância (km)')
    plt.ylabel('Semivariância')
    plt.title(f'Semivariograma target_log1p - {direction}')
    plt.tight_layout()
    plt.show()


## 4. Correlação espacial por limiar de dBZ

In [ ]:
for event_id in ['ge_20','ge_30','ge_40','ge_45']:
    t = spatial[(spatial.field.eq(event_id)) & (spatial.direction.isin(['EW','NS']))]
    plt.figure(figsize=(8,4))
    for direction, g in t.groupby('direction'):
        g = g.sort_values('distance_km')
        plt.plot(g.distance_km, g.pearson_r, marker='o', label=direction)
    plt.xlabel('Distância (km)')
    plt.ylabel('Correlação')
    plt.title(event_id)
    plt.legend()
    plt.ylim(-0.05,1.05)
    plt.tight_layout()
    plt.show()


## 5. Morfologia global

In [ ]:
morph = pd.read_parquet(OUT/'morphology_summary.parquet')
display(morph)


## 6. Morfologia sazonal

In [ ]:
season = pd.read_parquet(OUT/'morphology_by_season.parquet')
display(season[season.event_id.isin(['ge_30','ge_40','ge_45'])])


## 7. Diagnóstico de contato com borda

In [ ]:
t = morph[['event_id','border_touch_rate_given_event','largest_component_border_touch_rate_given_event']]
display(t)
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(t.event_id, t.border_touch_rate_given_event)
ax.set_ylabel('Taxa de patches com evento tocando borda')
ax.set_ylim(0,1)
plt.tight_layout()
plt.show()


## Interpretação

Os resultados descrevem a distribuição espacial dos patches sobrepostos que o CorrDiff vê durante o treinamento. Não trate as métricas como climatologia espacial de campo completo. Em especial, componentes que tocam a borda podem continuar fora do patch.